In [1]:
from sumo_rl.environment.env import SumoEnvironment

In [2]:
sumo_env = SumoEnvironment(
        net_file="single-intersection.net.xml",
        route_file="single-intersection-vhvh.rou.xml",
        use_gui=False,
        num_seconds=1000,
        yellow_time=3,
        min_green=5
    )

In [3]:
# traffic signal id
print(sumo_env.ts_ids)

['t']


In [4]:
### obs = [phase_one_hot, min_green, lane_1_density,...,lane_n_density, lane_1_queue,...,lane_n_queue]
### phase_one_hot is a one-hot encoded vector indicating the current active green phase
### min_green is a binary variable indicating whether min_green seconds have already passed in the current phase
### lane_i_density is the number of vehicles in incoming lane i dividided by the total capacity of the lane
### lane_i_queueis the number of queued (speed below 0.1 m/s) vehicles in incoming lane i divided by the total capacity of the lane

obs = sumo_env.reset()
print(obs)  

{'t': array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0.], dtype=float32)}


In [5]:
### The action space is discrete. Every 'delta_time' seconds, each traffic signal agent can choose the next green phase configuration.
### Important: every time a phase change occurs, the next phase is preeceded by a yellow phase lasting yellow_time seconds.

print(sumo_env.action_space)

Discrete(4)


In [6]:
import numpy as np
from gymnasium import spaces, Env

class SumoEnvWrapper(Env):
    def __init__(self, env):
        super().__init__()
        self.env = env

        example_obs = self.env.reset()
        flat_obs = self._flatten_obs(example_obs)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=flat_obs.shape, dtype=np.float32)
        self.action_space = env.action_space

    def _flatten_obs(self, obs):
        return np.concatenate([np.array(v).flatten() for v in obs.values()]).astype(np.float32) # No need to do this since we only have one agent

    def reset(self, **kwargs):
        obs = self.env.reset()
        return self._flatten_obs(obs), {} # return (obs, info)

    def step(self, action):
        ts_id = self.env.ts_ids[0]
        dict_action = {ts_id: int(action)}

        obs, reward, done, info = self.env.step(dict_action)
        flat_obs = self._flatten_obs(obs)

        if isinstance(reward, dict):
            total_reward = float(sum(reward.values()))
        else:
            total_reward = float(reward)

        if isinstance(done, dict):
            done_flag = any(done.values())
        else:
            done_flag = done

        return flat_obs, total_reward, done_flag, False, info

    def render(self):
        return self.env.render()

    def close(self):
        return self.env.close()


In [7]:
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

def make_env():
    wrapped_env = SumoEnvWrapper(sumo_env)
    monitored_env = Monitor(wrapped_env)
    return monitored_env

env = DummyVecEnv([make_env])

In [8]:
from sb3_contrib import ARS
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor


model = ARS(
    policy="MlpPolicy",
    env=env,
    learning_rate=0.03,
    n_delta=32,
    n_top=20,
    verbose=1,
    tensorboard_log="./ars_log/"
)

model.learn(total_timesteps=300000, tb_log_name="run2_C")

# model saving
model.save("ars_sumo_model_A")

# model testing
test_env = DummyVecEnv([lambda: Monitor(make_env())])
obs = test_env.reset()
done = False
total_reward = 0

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    total_reward += reward[0]

print(f"Total reward (test): {total_reward}")

Using cpu device
Logging to ./ars_log/run2_C_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -853     |
|    return_std      | 103      |
| time/              |          |
|    fps             | 24       |
|    time_elapsed    | 513      |
|    total_timesteps | 12800    |
| train/             |          |
|    delta_std       | 0.05     |
|    iterations      | 0        |
|    learning_rate   | 0.03     |
|    step_size       | 1.45e-05 |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -818     |
|    return_std      | 104      |
| time/              |          |
|    fps             | 25       |
|    time_elapsed    | 1003     |
|    total_timesteps | 25600    |
| train/             |          |
|    delta_std       | 0.05     |
|    iterations      | 1        |
|    learning_rate   | 0.03     |
|

In [9]:
new_env = SumoEnvironment(
    net_file="single-intersection.net.xml",
    route_file="single-intersection-vhvh.rou.xml",
    use_gui=True,  
    num_seconds=1000,
    yellow_time=3,
    min_green=5
)

wrapped_env = SumoEnvWrapper(new_env)

obs, _ = wrapped_env.reset()
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _, info = wrapped_env.step(action)
    wrapped_env.render()  